
# Modeling Guidelines for Form Design Files (FDFs)

This notebook ports narrative guidance from the *SDC TRG ^N4* reference into concise, implementation-focused notes. Section numbers mirror the original document so we can track coverage. Sections 1.1–2.1 already live in `Intro To SDC/Introduction.ipynb`; we continue here with the remaining material.



## 2.2 SDC Actors

SDC transactions hinge on distinct software actors plus the organizations that author content:

- **Form Managers (FMs)** host repositories of Form Design Files (FDFs), respond to Form Filler requests, and manage authentication, authorization, and issuance/enforcement of instance IDs and versions.
- **Form Fillers (FFs)** fetch FDFs (or pre-rendered HTML/URLs), render DEFs, enforce implicit/explicit rules, capture/validate user responses, and transmit completed FDF-Responses (FDF-Rs).
- **Form Receivers (FRs)** accept FDF-Rs, store captured data either as native SDC XML or "shredded" into local schemas, and may perform schema validation, Schematron checks, version control, patient matching, and authorization.
- **Form Creators** (e.g., CAP, NCI, CCO) design FDF content/behavior—either by editing XML directly or using tooling—and deliver finished FDFs to Form Managers even though they are not IHE transaction nodes.

Together these actors enable a pipeline in which any conformant Form Filler can render any conformant FDF, exchange captured data, and rely on Form Receivers to persist, validate, and route the results.



## 2.3 The SDC Information Model: Data Entry Forms and Data Elements

- **Data Entry Forms (DEFs)** appear in UIs (web, desktop, EHR) and contain data items—question/answer pairs that SDC formalizes as **Question/Answer Sets (QAS)**.
- **Form Design Files (FDFs)** are XML blueprints for DEFs. They standardize the QAS content, behavior, and nesting independent of rendering technology.
- SDC Data Elements (DEs) act as reusable QAS blocks that can live in both registries and FDFs, providing portable semantics tied to context.
- When a DEF captures responses, the resulting XML is still the original FDF plus responses, now termed an **FDF-Response (FDF-R)**.
- FDFs may also specify report layouts/text that differ from DEF presentation, so downstream systems can generate audience-specific summaries while preserving the original structure.

This information model ensures a single machine-readable schema governs design, rendering, storage, and redistribution of DEF content, allowing for full-fidelity round trips between actors.



## 3 Data Entry Forms

### 3.1 Introduction to Data Entry Forms

Each DEF/FDF describes six XML Form Components (XFCs): Section (S), Question (Q), ListItem (LI), DisplayedItem (DI), ButtonAction (BA), and InjectForm (IF).

- **Questions (Q)** and **ListItems (LI)** implement the QAS units that capture data; other components support layout, navigation, and behavior.
- DEF visual controls (widgets) are implementations of these XML components. Designers are free to theme controls, but the underlying IDs, attributes, and nesting come directly from the FDF.
- Components may embed metadata that control layout, behavior, validation, and reporting. Hidden nested Questions ("untitled" Questions) can be activated implicitly when their parent ListItem is selected, illustrating how hierarchy drives activation logic.
- Repeating Sections/Questions allow users to add multiple instances of the same XFC subtree, preserving structure and context for each repetition.

### 3.2 Reporting from DEFs

SDC anticipates that report-friendly text and layout often differ from data-entry wording. Key considerations:

- Reports may target multiple audiences (patients vs. specialists) or aggregate multi-source data, so FDF metadata include `reportText` properties to supply narrative optimized for output.
- Nesting information, numeric units, and other metadata from the FDF should guide report generation to avoid misinterpretation.
- While DEF layout can serve as an initial reference, implementers should treat reporting as a separate design activity that still reuses SDC context and metadata to keep semantics intact.



## 4 The Form Design File (FDF)

### 4.1 Additional Definitions

- **XML Form Components (XFCs)** are the SDC XML constructs for Sections, Questions, ListItems, DisplayedItems, InjectForm blocks, and ButtonActions. Every XFC carries a unique `ID` and optional attributes/sub-elements that describe behavior and presentation.
- A **Question/Answer Set (QAS)** is the combination of a Question and its permissible responses (fill-in values or ListItems). QAS definitions live in the FDF and are rendered by Form Fillers.
- **Controls/widgets** refer to the DEF-side UI objects that render XFCs. Although their presentation varies, their structure, IDs, and metadata originate from the FDF.
- **Metadata** covers the attribute values that drive rendering, behavior, storage, exchange, and reporting. **Data** refers to captured user responses (fill-in values or selected ListItems).
- **Context** is the hierarchical nesting established by the FDF XML. Maintaining context—especially ancestors—preserves semantics when storing, coding, or reporting responses.

### 4.2 Structural Overview

- SDC favors attribute-centric XML: element text content is minimized, with authoritative values stored in attributes (often `val`) or in sub-elements such as `Property` or `Comment`.
- Sections, Questions, and ListItems that own child components wrap them in a `ChildItems` container, which separates descendants from peer metadata. Questions wrap their ListItems inside `Question/ListField/List/ListItem` instead of `ChildItems`.
- Only Section, Question, and ListItem elements can contain nested XFCs (via `ChildItems`), while DisplayedItem, ButtonAction, and InjectForm cannot (though InjectForm can insert external XML that has its own descendants).
- Three specially named Sections—`Header`, `Body`, and `Footer`—sit directly under the root `FormDesign`. No other XFCs may be direct children of `FormDesign`.
- DisplayedItems provide textual content and styling hooks, but lack descendants and cannot reside inside repeating structures; use Sections if nested display content is required.
- Questions appear either as fill-in controls (with response fields typed/validated according to schema constraints) or as selection controls backed by ListItems.

These conventions let tooling and runtimes traverse an FDF predictably: IDs remain stable, nesting defines activation/visibility, and attribute-driven metadata keeps the XML concise while still expressive enough for automation.


### 4.2.1 SDC Conventions

- **Attribute-centric XML**: Elements avoid inner text; canonical values live in attributes such as `val`, which keeps the XML terse and makes it easy to add sub-elements (`Property`, `Comment`, etc.) without rearranging text nodes.
- **XPath notation**: Paths like `Question/ListField/List/ListItem/@ID` are used throughout this guide to reference nested elements and individual attributes without reproducing the entire XML block.
- **Schema vocabulary**: Every SDC element corresponds to an XML Schema complex Type whose name ends with `Type` (e.g., `QuestionType`). Discussions often reference either the element or its Type, so keeping the suffix straight matters when browsing the schema.
- **SDC datatypes**: Most primitives mirror W3C XML Schema datatypes but are wrapped in paired SDC types—`<name>_DEtype` (for data-entry fields, including facets like `maxExclusive`/`totalDigits`) and `<name>_Stype` (for designer-authored static values that do not accept user input). SDC also defines `xml` and `html` datatypes so authors can embed custom XML/XHTML; in those cases the appropriate namespaces and schemas (e.g., `xhtml.xsd`) must be supplied.


### 4.2.2 A First Look at FDF XML

- **FormDesign layout**: Every FDF is rooted in a `FormDesign` (or `DemogFormDesign`) element whose immediate children are exactly three Sections: `Header`, `Body`, and `Footer`. All visible content lives somewhere under those sections.
- **Namespace & schema declarations**: A conformant file declares the default SDC namespace (`urn:ihe:qrph:sdc:2016`), the usual `xsd`/`xsi` helper namespaces, and any optional namespaces (e.g., `xmlns:h="http://www.w3.org/1999/xhtml"` when XHTML fragments are embedded). `xsi:schemaLocation` can point to `SDCFormDesign.xsd` so validators can check structure.
- **Canonical attribute block**: The FormDesign element typically includes `order`, `type`, `styleClass`, and a human-friendly `name`, followed by identifier attributes such as `baseURI`, `lineage`, `version`, `ID`, `fullURI`, and `filename`. These values pinpoint the lineage/version of the template and hint at the expected storage name.
- **Properties and content**: Immediately inside `FormDesign` you will often see `Property` entries (e.g., `ShortName`, `ApprovalStatus`) that capture domain metadata, followed by the Header/Body/Footer sections and optional global containers such as `Rules`. Example snippets in the TRG show the entire structure in one glance so implementers can recognize the expected ordering of namespaces, attributes, properties, and structural sections.


### 4.2.3 The XFCs

**ChildItems and nesting**
- `ChildItems` wraps any descendant Sections, Questions, or ListItems, allowing those child elements to sit beside metadata (`Property`, `Comment`, `Event`) without ambiguity.
- The only exception is the `Question/ListField/List/ListItem` structure, which replaces `ChildItems` for answer lists.
- Only Sections, Questions, and ListItems can own `ChildItems`; DisplayedItems, ButtonActions, and InjectForms cannot, though an InjectForm can import an entire subtree from another FDF.

**Component quick-reference**
- **Section (S)**: Fundamental grouping block. Header/Body/Footer are special Sections directly below `FormDesign`; other Sections can nest arbitrarily, can repeat, and are used to shape both the DEF layout and report hierarchy.
- **DisplayedItem (DI)**: Pure display text or instructions with a unique `ID`, optional styling, and no descendants. Use Sections instead of DI whenever nested content or repeating behavior is needed.
- **Question (Q)**: Comes in two flavors. Question-Response (QR) items contain a `ResponseField/Response` element so the DEF captures typed input. Selection questions host a `ListField/List` of ListItems and never carry a `ResponseField` at the Question level. Questions may also nest additional Sections/Questions via `ChildItems` to model conditional sub-questions.
- **ListItem (LI)**: Represents answer choices. Simple LIs capture a discrete selection, whereas `ListItemResponse` entries include a `ListItemResponseField/Response` so a choice can trigger an additional fill-in area (often called LIR/answer fill-in). All ListItems for a Question live inside the same `ListField/List` wrapper, and specific metadata (e.g., selection activation) is carried on the LI itself.
- **ButtonAction (BA)**: An explicit user-triggered control that raises events or rules (e.g., "Save draft", "Add section"). While any XFC can fire actions, ButtonAction reserves a visible widget dedicated to that purpose.
- **InjectForm (IF)**: Placeholder used to drop in other SDC content—anything from a snippet to an entire FDF—at runtime. The injected XML behaves as if it were authored inline, enabling library-style reuse of sections or template fragments.


### 4.3 FormDesign Attributes and Properties

The FormDesign root carries three classes of metadata:

1. **Static attributes** entered by the form designer (identity, filenames, display titles).
2. **Instance-tracking attributes** that Form Managers/Form Fillers populate when a specific DEF instance is created or edited (unique URIs for the form instance and its revisions).
3. **Instance status attributes** that describe the clinical workflow state of the captured data (approval/completion flags, new/changed data indicators).

Understanding which actor sets each field—and when—is essential for interoperable round trips between FM, FF, and FR systems.


#### 4.3.1 FDF Namespaces

- Every FormDesign declares the default SDC namespace `xmlns="urn:ihe:qrph:sdc:2016"` and typically lists `xmlns:xsd` and `xmlns:xsi` as helpers for schema references.
- Additional namespaces are included only when needed (for example, `xmlns:h="http://www.w3.org/1999/xhtml"` when embedding XHTML content or custom namespaces for injected XML).
- `xsi:schemaLocation` may point to the local `SDCFormDesign.xsd` so validators know where to fetch the schema definition used to check an FDF or FDF-R.


#### 4.3.2 FormDesign Attributes

**Static attributes (designer supplied)**
- `formTitle`: User-facing title that can appear in form pickers or the DEF header.
- `baseURI`: Required identifier for the organization or content domain responsible for the template; serves as the root for derived IDs/URIs.
- `basedOnURI`: References a template that this form extends, letting tooling trace customizations back to a standard FDF.
- `lineage` / `version`: Text identifiers that group all revisions of a logical form and label the current release.
- `ID`: Conventionally `lineage_version_sdcFDF`, ensuring every empty FDF (no captured data) has a predictable identifier.
- `filename`: Suggested storage name (often `lineage_version_sdcFDF.xml` for empty forms and `lineage_version_instance_instVer_sdcFDFR.xml` for responses).
- `fullURI`: Query-string style concatenation of `_baseURI`, `_lineage`, `_version`, and `_docType=sdcFDF`. Each component is URL-escaped so the URI can double as a stable identifier.
- `prevVersionURI`: Optional pointer to the immediate prior release so systems can diff or fetch it automatically.

**Instance attributes (populated on FDF-Rs)**
- `formInstanceURI` *(FM or FF)*: Unique URI for a specific filled-out form. Format mirrors `fullURI` but adds `_instance=<GUID>` and stays constant across edit sessions.
- `formInstanceVersionURI` *(FM or FF)*: Identifies a single save/edit of that instance (`_instVer` often encodes a timestamp). Must change every time the user saves data.
- `formPreviousInstanceVersionURI` *(FM or FF)*: Carries the prior `formInstanceVersionURI`, enabling systems to compare revisions.

**Instance status attributes (FF assigned with user input)**
- `approvalStatus`: Workflow state such as `inProcess`, `preliminary`, `approved`, `cancelled`, or `retracted`.
- `completionStatus`: Indicates whether the requested information is `pending`, `incomplete`, or `complete`.
- `newData` / `changedData`: Boolean indicators that flag whether a package/form/section/question now contains new or modified responses relative to the previous submission.


#### 4.3.3 FDF Identifiers

- Each FormDesign has one `ID` per lineage/version combination, and every XFC inside the form has its own `ID` that is unique within that FDF but may be reused in other forms.
- Long GUID-style IDs are technically possible but discouraged because they are hard to read and prone to copy errors; instead SDC relies on `baseURI` + `ID` combinations to form composite globally-unique identifiers (CGUIs).
- `baseURI` defaults to the FormDesign value and is inherited by descendants unless explicitly overridden. Override it only when an embedded XFC originates from a different steward so the CGUI correctly reflects authorship.
- Organizations should use stable, registered identifiers (domain names or GUIDs) for `baseURI`, ideally in URL form but without the protocol prefix to avoid unnecessary length or validation issues.


#### 4.3.4 FormDesign Properties (eCC)

- `Property` elements under FormDesign let template authors attach arbitrary metadata (`name`, `type`, `propName`, `val`, `order`, optional `styleClass`). FFs may render them, suppress them, or use them purely as hints depending on the workflow.
- The CAP eCC library demonstrates common property families: copyright notices, generic header text, organ system categories, official/short names, protocol names & versions, template IDs, usage restrictions, regulatory flags (`CAP_Required`), publication/ accreditation dates, approval statuses (e.g., `RC1`, `RC2`), and tumor-staging references such as `AJCC_Version`.
- Because properties live in the XML, downstream tools can read or display them without relying on external registries, and form maintainers can extend the metadata vocabulary without altering the core schema.


## 5 Introduction to SDC Basic Schema Types

The remainder of the TRG pivots from actor/process concepts to the schema mechanics that make FDFs interoperable. The SDCFormDesign schema set is deliberately modular and heavily relies on inheritance so that tooling can generate object models and validators that behave consistently across every XFC.


### 5.1 SDC Schema File Overview

- The design stack is a chain of XSDs where each level `xs:include`s the one beneath it: `SDCFormDesign.xsd → SDCExpressions.xsd → SDCResources.xsd → SDCDataTypes.xsd → SDCBase.xsd`.
- Retrieval flows bring in additional files (`SDCRetrieveForm.xsd` plus `SDCTemplateAdmin.xsd` and `SDCMappings.xsd`), while submission payloads rely on `SDCSubmitForm.xsd`, which itself includes `SDCFormDesign.xsd`.
- For modeling guidance we stay focused on `SDCFormDesign` and its subordinate schemas because they define the structure of FDFs and FDF-Rs; the retrieval/submit shells simply layer transport metadata on top.


### 5.2 Schema Files and Type Hierarchy (5.2 & 5.2.1)

- Schema layering (the XSD include chain) is different from the **type** hierarchy. Types are declared across the files and derived from progressively more capable ancestors—similar to an OO inheritance tree.
- All SDC datatypes mirror W3C datatypes, with additional SDC-specific wrappers (e.g., `_DEtype` vs `_Stype`) and special `xml`/`html` types for embedding structured content with their own namespaces.
- Example 6 in the TRG outlines the major abstract bases (`BaseType`, `ExtensionBaseType`, `IdentifiedExtensionType`, `DisplayedType`, `RepeatingType`) along with the concrete XFC types (`SectionItemType`, `QuestionItemType`, `ListItemType`, `ButtonItemType`, `InjectFormType`). Remember that the schema keeps element names PascalCase while datatype elements stay camelCase to align with W3C conventions.


### 5.3 SDC Schema Inheritance Model and XFC Definitions

- All XFCs ultimately inherit from `BaseType` and `IdentifiedExtensionType`; most also pass through `DisplayedType`, and Sections/Questions add `RepeatingType` so they can repeat instances at runtime.
- Because the inheritance tree pushes attributes downward, implementers must understand which capabilities come "for free" with each ancestor (e.g., `DisplayType` contributes links/blobs/events; `RepeatingType` contributes `repeat`, `minCard`, `maxCard`, `instanceGUID`).
- The inheritance diagrams also explain why FormDesign and DataElement share behavior: both derive directly from `IdentifiedExtensionType`, ensuring ID/baseURI semantics line up with the XFCs they host.


### 5.4 BaseType (abstract)

All higher-level elements inherit these optional attributes:

- `name`: W3C `xs:ID` identifier intended for programmatic references; keep it syntax-friendly for target languages.
- `type`: Space-delimited tokens (W3C `NMTOKENS`) defined per implementation guide to flag behaviors such as `tooltip` or `statusLineText`. Style hints belong in `styleClass` instead.
- `styleClass`: CSS-like tokens pointing at visual styles (e.g., `alignTopLeft`, `pageBreak-after`). Multiple values are allowed.
- `order`: Decimal used to retain a canonical ordering even if runtime collections reorder nodes. Useful whenever tooling cannot rely on raw XML order.


### 5.5 ExtensionBaseType (abstract)

`ExtensionBaseType` (EBT) enriches descendants with three powerful extensibility hooks—`Property`, `Comment`, and `Extension`. Almost every SDC type inherits from EBT (datatypes are the notable exception), which means you can attach domain-specific metadata, comments, or foreign XML wherever needed without modifying the base schema.


#### 5.5.1 Property Elements

- `Property` elements (instances of `PropertyType`) can nest arbitrarily because Properties themselves inherit from EBT. This allows complex metadata trees under any XFC.
- A Property carries `propName`, optional `type/styleClass/order`, and either a simple `val` or a strongly typed `TypedValue` block. `TypedValue` can hold any SDC datatype, including `html`/`xml`, which in turn require their own namespaces and schema references.
- Common patterns:
  - Use `TypedValue/html` for author-supplied rich text while keeping `val` for plain strings.
  - Store dates or other constrained values in typed children (e.g., `<date val="2019-01-01"/>`) so validation can catch formatting issues.
  - Provide alternate text for reports with `propName="reportText"`, optionally setting it to `{no text}` to suppress DEF titles entirely in generated reports.
  - Supply alternate DEF text via `altText`, or define HTML versions of a title via `titleHTML` to inject markup without abandoning the standard `title` attribute.
- Because Properties appear on any EBT-descended element, they are the recommended place to carry localized labels, regulatory annotations, or workflow hints that would otherwise require schema changes.


#### 5.5.2 FormDesign Property Patterns

Building on the CAP eCC examples from §4.3.4, section 5.5.2 highlights advanced usage:

- Reports can mix-and-match Property-driven text (e.g., DisplayedItems with `reportText` become "report notes" even when the on-screen title is blank).
- Properties can deliver parallel representations of the same concept—plain `val` for storage, `TypedValue/html` for rich rendering, or `TypedValue/date` to enforce ISO-8601.
- Designers and implementers must coordinate on `propName` vocabularies so that FFs know which metadata to render, ignore, or treat as machine-only hints.


#### 5.5.3 Comment Elements

- `Comment` elements inherit from `BaseType` and therefore accept `type`, `styleClass`, and `order`. Their `val` holds plain-text notes (no markup) supplied by designers, implementers, or even end users.
- Allowing DEF users to create comments implies the UI must expose affordances (icons, inline buttons, etc.) and the backend must validate where those comments are permitted (e.g., on particular Questions or ListItems).


#### 5.5.4 Extension Elements

- `Extension` blocks are the sanctioned escape hatch for non-SDC XML. Each `Extension` child must live in its own namespace—and ideally ship with a schema—so validators can distinguish foreign content.
- Extensions offer virtually unlimited customization but require governance: form designers and implementers need to agree upfront on which namespaces are supported, where they may appear, and how to validate/process them to avoid interoperability regressions.


#### 5.6 IdentifiedExtensionType (abstract)

- Adds the required `ID` attribute plus the optional-but-inherited `baseURI` to every descendant. Together they let implementers construct composite globally unique identifiers (CGUIs) such as `baseURI + ID` per lineage.
- `baseURI` is required on `FormDesign` and inherited by all XFCs unless overridden. Override it only when an injected component comes from another steward so traceability is preserved.
- Keeping `baseURI`/`ID` pairs stable across versions signals that the semantic meaning of an XFC hasn’t changed, which is critical for downstream data stores comparing FDF versions.


#### 5.7 DisplayedType

- Serves double duty: it defines the standalone `DisplayedItem` XFC and acts as the base type for the visible controls (Section, Question, ListItem, ButtonAction).
- Capabilities inherited from DisplayedType include hyperlinks (`Link`), binary payloads (`BlobContent` with `Hash`, `BlobURI`, `BinaryMediaBase64` children), `CodedValue` structures, and core events (`OnEnter`, `OnExit`, `OnEvent`) plus Guards (`ActivateIf`, `DeActivateIf`).
- Because DisplayedType components lack `ChildItems`, use Sections when hierarchical nesting is required. InjectForm does not inherit from DisplayedType—it relies on the injected content for visual behavior.


#### 5.8 RepeatingType (abstract)

- Extends DisplayedType with runtime repetition metadata: `minCard`, `maxCard`, `repeat` (flag to allow user-controlled instances), and GUID bookkeeping (`instanceGUID`, `parentGUID`).
- Only SectionItemType and QuestionItemType derive from RepeatingType, which is why only Sections/Questions can be configured as repeatable blocks that replicate their entire subtree.
- These attributes underpin features like nested repeats and instance tracking, so they are essential when building DEF UIs that add/remove rows on demand.


## 6 The XFCs

The remaining TRG content walks through each XML Form Component (XFC) in depth. Every XFC inherits the attribute stacks described in Chapter 5, but their concrete behavior depends on how Form Fillers interpret IDs, nesting, events, and repeat metadata.


### 6.1 XFC Identifiers and Names

- Every XFC must carry a unique `ID` within its FDF; schema validation enforces uniqueness, but implementers must preserve IDs when repeating or cloning nodes in a DEF.
- Optional `name` attributes (from `BaseType`) act as programmer-friendly identifiers. They should follow language-friendly conventions (letters/underscore prefixes, ASCII-only) to avoid issues when auto-generating code or binding to scripting environments.
- When DEF controls inherit the same IDs/names as their XML counterparts, Form Fillers can round-trip user responses back into the matching XML nodes with minimal mapping logic.


### 6.2 The DisplayedItem XFC

- DisplayedItems provide instructional or narrative text anywhere in the form. They require unique IDs/titles but cannot own `ChildItems`; for nested display content, switch to a Section.
- Typical uses include inline notes, report-only annotations (set `title=""` and supply `Property propName="reportText"`), or alternative renderings via `titleHTML`.
- Because DisplayedItems inherit DisplayedType, they can host Blobs, Links, CodedValues, and events/guards just like Questions or Sections.


#### 6.2.1 DI Substructure

Key child elements inside a DisplayedItem:

- **BlobContent**: Embed base64 media along with optional `Hash`, `BlobURI`, and `BinaryMediaBase64` children pointing to the source or providing integrity checks.
- **Link**: References internal or external resources; UI implementations decide whether links open inline, in dialogs, or new tabs.
- **CodedValue**: Carries hidden coding metadata (code, text, match degree, code system info, release/version/OID/URI). Useful when a display note must also be traceable to terminology content.
- **Properties**: Use `reportText`, `altText`, `titleHTML`, etc., to decouple on-screen wording from report wording or to add rich formatting.


### 6.3 The Section XFC

- Sections wrap related content and can contain any mix of Sections, Questions, DisplayedItems, ButtonActions, or InjectForms (ListItems must still live inside Questions).
- Header/Body/Footer are special Sections directly under FormDesign; other Sections may be nested arbitrarily and can be configured as repeatable via RepeatingType attributes.
- Use Sections whenever you need to group metadata, apply shared properties/events, or control activation/visibility for an entire subtree.


### 6.4 The Question XFC

- Each Question defines a QAS rendered as one or more controls in the DEF. The Question’s `title` drives the on-screen label, while properties (e.g., `reportText`) can swap wording for reports.
- Questions either capture responses directly (`ResponseField/Response/<datatype>`) or host a `ListField/List` of ListItems (single- or multi-select). They can also own nested Sections/Questions via `ChildItems` to handle conditional follow-ups.
- IDs must remain unique even when repeating Questions; Form Fillers that instantiate repeats need to manage suffixed IDs while preserving lineage for reporting.


#### 6.4.1 Question-Response (QR)

- QR Questions omit ListItems and instead provide a `ResponseField` that encloses a `Response` typed with any SDC datatype (string, integer, date, html, etc.).
- Validation facets (e.g., `maxLength`, `maxInclusive`, `pattern`, `mask`) live on the datatype element, so when a user enters a value the Form Filler writes it to the datatype’s `val` attribute and enforces the constraints.
- Supplementary metadata such as `TextAfterResponse` or `ResponseUnits` convey symbols/units that should appear alongside the input and in generated reports (units default to UCUM identifiers).
- Because datatypes use lower camelCase (per W3C), be mindful when scanning XML for response content—it lives under elements like `<integer>` or `<dateTime>` rather than capitalized names.


#### 6.4.2 Single-Select Questions (QS)

- A QS hosts a `ListField/List` of ListItems with `maxSelections="1"` (the default). Form Fillers typically render radio buttons or dropdowns to enforce single choice.
- Default selections can be specified in the FDF by setting `selected="true"` on a ListItem; the FF preselects that option when rendering the DEF.
- When additional input is needed for an answer, nest a `ListItemResponseField` under that ListItem so the user can provide a fill-in response after selecting the option.


#### 6.4.3 Multi-Select Questions (QM)

- Multi-select Questions set `ListField/@maxSelections` to `0` (unlimited) or any number >1, enabling checkboxes or multi-select pickers.
- Each selected ListItem is persisted by setting its `selected` attribute to `true`; unselected items either omit the attribute or keep it `false`.
- Because multi-select Lists can become long, leverage DisplayedItems or Sections to group subsets and keep activation rules manageable.


#### 6.4.4 Capturing User Responses

SDC captures data in two ways:

1. **Response values** – QR Questions and ListItems with `ListItemResponseField` write their user input into the nested datatype element’s `val` attribute (e.g., `<string val="answer"/>`).
2. **Selections** – For QS/QM, each chosen ListItem gets `selected="true"`. Defaults can be pre-populated using the same attribute so the DEF loads with recommended answers.

This dual model lets DEFs mix fill-in fields with list selections, and Form Fillers simply mirror the XML structure when saving responses.


#### 6.4.5 Response Metadata

- Datatype elements include validation facets such as `maxLength`, `minLength`, `maxInclusive`, `mask`, `pattern`, `allowGT`, etc. When the user enters a value, FFs must enforce these constraints before writing `val`.
- `TextAfterResponse` displays fixed text (e.g., `%`) immediately after the input; `ResponseUnits` records units (default unit system UCUM) so reports can reproduce the correct notation even if the UI hides it.
- These metadata originate in the FDF, so validation and reporting logic should read them directly rather than hard-coding limits in the UI.


### 6.5 DEF Helper Components

Beyond Questions, FDFs define helper components—DisplayedItems, Sections, Properties, media blobs—that organize the layout and supply context cues. They inherit the same property/event infrastructure as Questions, so implementers can:

- Insert explanatory text (DisplayedItems) or hierarchical containers (Sections) to break complex forms into digestible areas.
- Attach properties such as `reportText`, `titleHTML`, or custom metadata to control how helper content appears in the DEF versus generated reports.
- Use helper components to anchor activation logic: for example, wrap a cluster of Questions in a Section and apply guards/properties once rather than repeating metadata on each Question.


## 7 DEF Functional Considerations

Chapters 2–6 describe the structure of FDFs; Chapter 7 explains how Form Fillers should behave when rendering, validating, and persisting those structures. The key theme: the DEF must faithfully maintain the FDF XML, applying activation/visibility/requirement rules exactly as encoded in the metadata.


### 7.1 The DEF Maintains and Manipulates the FDF

- Form Fillers keep an in-memory copy of the FDF, updating it with user responses as the session progresses.
- Each user action either writes values into `ResponseField` elements or toggles attributes like `selected`, ensuring the DEF and FDF stay synchronized.
- When the DEF saves or submits, it serializes the modified FDF (creating an FDF-R) so downstream actors receive the exact XML with captured data.


### 7.2 Implied Activation (IA)

- Activation means a user can see and interact with an item; deactivated items are disabled/hidden and must not be validated, reported, or saved even if they held prior data.
- The hierarchy controls activation: selecting a ListItem or answering a Question implicitly activates its descendants; deselecting/deleting the parent response deactivates the subtree and clears it from reporting.
- Implementers should default to activating only the outermost sections/questions on first load, progressively revealing deeper levels as the user answers upstream controls.


#### 7.2.1 Complex Nesting & Invisible Sub-Questions

- Nested activation can span multiple levels (e.g., a ListItem activates a child Question, whose response activates another Question). DEFs must track these dependencies so that removing any parent answer reverses all descendant activations.
- Invisible Questions (no `title`) often act as structural placeholders or conditional follow-ups; their `altText` properties can guide developers but should be displayed only if users need the extra context.


#### 7.2.2 Complex Item Dependencies

- Activation may be driven by guards/events beyond simple parent-child relationships (covered later in Rules). Regardless of the trigger, the rule remains: a deactivated node and all descendants must be treated as unanswered.
- When designing DEF UIs, consider visual cues (e.g., fading sections, disabling controls) to signal activation state to users, matching the semantics encoded in the FDF.


### 7.3 Invisible Question Text

- Questions without a visible `title` still carry IDs and properties; their `altText` property provides a terse internal label for developers or database queries.
- Avoid showing `altText` directly to users unless stakeholders approve—its purpose is primarily diagnostic, especially when invisible Questions are activated under specific ListItems.


### 7.4 The `@mustImplement` Attribute

- Default behavior (`mustImplement="true"` or missing) means an XFC must appear in the DEF; setting `mustImplement="false"` lets implementers omit that control entirely (commonly used for optional ListItems or region-specific sections).
- Even in optional templates, required XFCs remain required—`mustImplement` governs visibility/implementation, not whether the user must answer once the control exists.


#### 7.4.1 Optional ListItems

- A required Question can still contain optional ListItems by marking individual items with `mustImplement="false"`; DEFs may hide those items unless the use case demands them.
- Implementation shortcuts (like prefixing optional items with `+`) are stylistic conventions; the authoritative indicator remains the XML attribute.


### 7.5 Required Responses

- `minCard` > 0 on a Question indicates the user must provide at least that many answers; `minCard=0` makes the Question optional.
- `responseRequired="true"` on a `ListItemResponseField` forces the user to fill in the associated value whenever that ListItem is selected—even if the parent Question is optional.
- Visual cues (asterisks, plus signs) should reflect these metadata so users know which controls must be answered.


### 7.6 Conditionally Required (CR) Behaviors & Reporting

- Some Questions/Sections are required only under certain conditions (e.g., a ListItem selection). Implementers often prefix such items with `?` to signal conditional status, but the definitive logic comes from FDF metadata and rules.
- CR items may need to be omitted from reports when the triggering condition isn’t met, even if the control was implemented. Reporting engines should inspect the same metadata used by the DEF to decide whether to include or suppress values.
- Always ensure that items with `mustImplement="true"` are present in the DEF, even if their responses might be suppressed in the final report based on conditional logic.


#### 7.6.1 “?” Prefix Display Model

- Some organizations prefix conditionally required (CR) Questions with `?` in the `title` so users know the response may be omitted from reports under certain conditions.
- Only Questions carry the `?`; ListItems keep normal titles. When using this convention, always provide a `reportText` property without the prefix so reports display clean labels.
- Alternate visual cues are acceptable as long as stakeholders agree—they just need to communicate that the item may be omitted downstream.


#### 7.6.2 Omit When Unanswered (OWU)

- Metadata: `mustImplement="true"` and `minCard="0"`. The Question must appear in the DEF, but the user decides whether the conditional requirement applies.
- Typical wording includes “if known” or “required only if…”. If unanswered, the Question is simply not reported (same as any unanswered Question), but if applicable it should be answered for clinical completeness.


#### 7.6.3 Omit When Selected (OWS)

- A ListItem with `omitWhenSelected="true"` signals that selecting it should suppress the entire Question (and descendants) from reports—useful for choices like “Not applicable” or “Specimen absent”.
- OWS Questions are still required to be answered; DEF validation should warn if the user skips them. Accreditation reviewers see only the final report, so omission is acceptable as long as the metadata indicates the item was intentionally suppressed.


### 7.7 Contiguity of ListItem Lists

- Inside `Question/ListField/List`, only ListItems and DisplayedItems (“list notes”) are allowed—Sections, Questions, ButtonActions, and InjectForms cannot appear directly and must be placed after the closing `</Question>` or under a ListItem’s `ChildItems`.
- When nesting XFCs beneath a ListItem, wrap them in `ChildItems`. Never insert a bare ListItem outside the ListField/List structure.


### 7.8 The Null Check Box

- SDC does not support tri-state checkboxes; ListItems are either selected (`true`) or not (`false`).
- If a Question requires options like Present/Absent/Unknown, model it as a single-select Question with mutually exclusive ListItems rather than relying on “null” checkbox states.


### 7.9 The Single Check Box

- A Question with a single checkbox is modeled as a QM (set `maxSelections="0"`) so the checkbox can be toggled on/off; QS controls (combo boxes/radios) generally lack this invertible behavior.
- When such a Question is required (`minCard>0`), an unchecked box is still a valid answer (“false”) and should pass validation.


#### 7.9.1 Reporting from the Single Check Box

- If the checkbox is unchecked, omit the Question and its ListItem from the report (and typically any descendants). When checked, include the Question/ListItem text plus any nested content.
- Activation of sub-Questions depends on guard attributes like `selectionDisablesChildren`, so reporting engines must respect those same rules.


#### 7.9.2 The Locked QAS

- Set `readOnly="true"` on a Question to lock it. Often paired with default selections or responses that capture invariant facts (e.g., specimen always present).
- Locked items may be hidden in the DEF, but their data should still be stored so queries can rely on the consistent QAS structure. Common pattern: `mustImplement=true`, `minCard=0`, empty `title`, and `reportText="{No text}"` with optional `altText` for internal reference.


### 7.10 Flavors of Unanswerable (FOU)

- Provide explicit ListItems for cases where a Question cannot be answered (e.g., “Cannot be determined”, “Not applicable”, “Unknown”). Some implementations make these ListItems LIRs so users can explain why.
- FOU choices are preferable to leaving Questions blank because they distinguish “not assessed” from “negative”.


## 9 Repeating Sections and Questions

SDC allows users to add additional Section or Question instances at runtime. Repeatability is controlled by `maxCard`: default `1` means no repeats, values greater than `1` set an upper bound, and `0` means unlimited. `minCard` still governs how many instances must exist, so a repeatable Section with `minCard=2` must capture at least two instances before validation passes.


### 9.1.1 Managing Repeated IDs

- Every time a user adds a repeated block, the Form Filler assigns a shared suffix `__#` to the `ID` (and optional `name`) of the repeated XFC and all descendants. The counter starts at `0` and increments for each new repeat anywhere in the form.
- Example: the first repeated instance uses `__1`; nested repeats inside that block share the same suffix. When another nested repeat occurs, the counter increments again (e.g., `__2`).
- This approach keeps IDs unique across all instances while preserving a clear link back to the original XFC definition.


### 9.1.2 Nested Repeats

- Nested repeat blocks follow the same rule: whenever a repeat occurs inside an already repeated block, the global counter increments and all IDs in the nested block adopt the new suffix.
- Original (non-repeated) XFCs remain unsuffixed, but implementers may treat the initial instance as `__0` conceptually to simplify code.
- Remember to apply the suffix to the `name` attribute as well if it is present; this ensures programmatic references stay aligned with the stored XML IDs.


## 10 DEF Validation and Reporting Results

Even with well-designed metadata, DEFs must be validated before submission and rendered into reports that meet accreditation requirements. Chapter 10 outlines how FFs should treat incomplete data and how reports should consume the captured responses.


### 10.1 Incomplete / Invalid DEF

- A DEF is not considered complete unless all applicable required Questions (minCard > 0) and required ListItemResponseFields (`responseRequired="true"`) have answers.
- Even so, FFs should allow users to save incomplete forms for later completion. Validation should flag missing/invalid data but never prevent saving.
- “Applicable” means reachable in the QAS tree: deactivated branches (due to activation logic) are excluded from completeness checks.


### 10.2 Validating, Saving, and Reporting

- Validation layers should warn users whenever required answers are missing or typed incorrectly, but they must not block the ability to persist the current state.
- Reports should omit unanswered Questions/ListItems by default (even if they are required) and should respect metadata such as `reportText` or `{No text}` overrides.
- Because accreditation reviewers focus on reports, ensure the report output clearly differentiates captured values while hiding conditional items that were legitimately omitted (e.g., OWU/OWS conditions).


### 10.3 Validation & Reporting Test Template

- Establish test scenarios that verify: required-item warnings, the ability to save with unresolved warnings, and correct suppression/inclusion of responses in reports (including OWU/OWS behaviors).
- Tests should confirm that report text uses `title` unless overridden by `reportText`, that multi-select outputs follow the desired formatting, and that sections or report notes appear only when populated.


### 11 Generating Reports from a DEF (Pointer)

While Chapter 10 focuses on validation mechanics, the following chapter expands on report design: ordering, formatting multi-select Questions, handling DisplayedItems, and tailoring text for synoptic formats. Use Chapter 10’s guidance as the baseline for when items should appear in reports, then apply Chapter 11’s layout best practices to meet stakeholder requirements.


## 11 Generating Reports from a DEF

Captured responses ultimately feed reports that clinicians, auditors, or regulators read. Chapter 11 explains how to turn DEF data into clear, compliant output without simply mirroring the on-screen layout.


### Report Text vs. DEF Text

- Use `title` for on-screen wording, but rely on `reportText` to customize what appears in the report; set `reportText="{No text}"` to suppress an item entirely.
- Avoid printing “?” prefixes or other UI-specific cues—reports should reflect the final, polished wording approved for clinical use.


### Layout Considerations

- Report ordering does not have to match the DEF hierarchy; choose layouts (columns, punctuation, line breaks) that maximize clarity for the target audience (e.g., CAP synoptic requirements).
- Multi-select Questions can be rendered as comma-separated lists or separate lines depending on readability, especially when some ListItems have fill-in responses.
- Include Sections or DisplayedItems (“report notes”) only when they add context; if a Section contains no reported content, omit it by default.


### Handling Unanswered Items

- Unanswered Questions/ListItems should be omitted unless the use case mandates showing them; this aligns with the validation rules from Chapter 10.
- CR behaviors (OWU/OWS) dictate when items are suppressed even if required—reports must respect those metadata.


### Synoptic & Accreditation Needs

- CAP synoptic formats often demand specific sectioning and wording; leverage helper components, reportText overrides, and consistent punctuation to meet those standards.
- Remember that the same responses may appear in multiple report sections (e.g., short diagnostic summary plus detailed section). Ensure the data is consistent wherever it shows up.


### Practical Tips

- Test report output alongside validation: ensure multi-select formatting, unit display, and suppression rules behave as expected.
- Use rich text or tables when appropriate, but keep accessibility and downstream parsing needs in mind.
- Treat DisplayedItems designed solely for reports as first-class metadata—store them in the FDF so report engines can include them without custom code.


## 12 SDC Instance Metadata

Before a form is distributed or submitted, it is wrapped in an `SDCPackage` that conveys package-level metadata, optional demographic forms, and the main `FormDesign`. Instance metadata explains how packages, forms, and submissions are uniquely identified across systems.


### SDCPackage / XMLPackage

- `SDCPackage` contains attributes such as `packageID`, the default SDC namespace (`xmlns="urn:ihe:qrph:sdc:2016"`), and `xsi:schemaLocation` so validators know which schema to use.
- The child `XMLPackage` holds one or more FDFs (e.g., `FormDesign` plus optional `DemogFormDesign`). Each FDF maintains its own `@ID` and inherits `baseURI`, enabling version/lineage tracking even when multiple forms ship together.
- Demographic content can be shared across domain-specific forms by packaging a reusable `DemogFormDesign` alongside each specialized FormDesign.


### Submission & Instance Tracking

- When a form instance is created, the metadata described earlier (`formInstanceURI`, `formInstanceVersionURI`, etc.) ride along with the package so Form Receivers can track specific submissions.
- Packaging metadata also helps downstream tooling know which schema version and template release were used, which is essential when validating data or reconciling updates.


### Versioning Overview

- CAP eCC forms use a structured version identifier (e.g., `123.456.789.1000043`), where each segment reflects the severity of changes (major CCP overhaul, ID changes, minor typos/schema tweaks).
- File names typically embed a shortened template name plus the version (e.g., `BreastCompleteExcisionand_2.000.011.xml`), and the same version string appears in the FormDesign header.
- When content changes introduce new/deprecated IDs, the second segment increments; minor text tweaks or schema-only updates adjust the third segment’s sub-components. This ensures every release can be uniquely referenced even if the underlying clinical content hasn’t changed.


## 14 Pre-population of SDC Forms

SDC supports two approaches for injecting existing data into a form: **pre-population** (import from external sources) and **auto-population** (data embedded directly in the FDF). Both rely on SDC maps to align external data elements with specific form fields.


### Pre-population (Pre-pop)

- External systems export structured data (often CDA or FHIR resources) plus an SDC map file. The Form Filler uses that map to populate corresponding fields before the user opens the form.
- Pre-pop is ideal when the FF needs to pull data from centralized EHRs or registries and align it with the latest released template.


### Auto-population (Auto-pop)

- Data is embedded directly in the FDF itself (e.g., via Properties or default responses) so the FF can fill fields automatically when rendering the form.
- Useful for local defaults or site-specific data that should always appear without contacting external services.


### SDC Maps

- Mapping files describe how external data fields (from CDA, FHIR, etc.) correspond to individual Questions/ListItems/ResponseFields in the SDC form.
- Maintaining accurate maps ensures pre-pop/auto-pop stays synchronized with template updates; when IDs change, maps must be revised accordingly.


## 16 IHE SDC REST API

SDC forms can be exchanged via the IHE REST interface, which complements the SDC packaging and submission workflows. The API exposes endpoints for retrieving blank forms and posting captured instances, each using the SDC XML format.


### 16.1 Empty SDC Forms

- Clients request blank forms from a Form Manager by supplying identifiers (e.g., package ID, form ID, or lineage/version). The response payload is the `FormDesign` XML (and optional `DemogFormDesign`) wrapped per Chapter 12/13.
- REST transport ensures the requester receives the current template along with the necessary metadata to populate or render it offline.


### 16.2 SDC Instance Forms

- Completed forms (FDF-R) are POSTed back to Form Receivers using the same REST profile. The payload includes the captured responses plus instance metadata (`formInstanceURI`, version URIs, etc.).
- Servers validate the submission against the SDC schema, process business rules, and store the data or forward it to downstream systems.
- This REST API aligns with IHE profiles (e.g., RFD) so SDC implementers can integrate with existing transport infrastructure while keeping the XML payloads consistent.


## 18 Versioning

SDC templates, packages, and eCC releases follow structured versioning schemes so implementers can track content changes, technical updates, and lineage relationships.


### 18.1 FDF Versioning

- FDF filenames embed a shortened template name plus a version identifier (e.g., `BreastCompleteExcisionand_2.000.011.xml`). The same version string appears in the FormDesign header.
- Increment rules: higher-order digits change for major revisions (new protocol or staging edition); middle digits change when Question/ListItem semantics or IDs change; lower digits handle minor text tweaks or schema-only updates.
- When IDs change (add/remove Questions, modify semantics), the middle segment increments and affected elements are deprecated/recreated with new IDs.


### 18.2 Package Versioning

- `SDCPackage` releases may carry their own identifiers (e.g., `packageID`) to track bundles of related forms. When a package includes updated FormDesigns or demographic forms, the package ID/version should change accordingly.
- Packaging version metadata ensures receivers can validate they have the correct combination of templates, demographic forms, and maps.


### 18.3 eCC Versioning

- CAP eCC templates use a structured identifier (e.g., `123.456.789.1000043`).
  - Segment 1 (major) changes for large CCP revisions (new staging, rewritten protocol).
  - Segment 2 tracks `@ID` changes (adding/deprecating Questions/ListItems).
  - Segment 3 captures minor content tweaks or schema-only updates (with sub-digits distinguishing XML-only changes).
- `1000043` denotes the namespace portion commonly omitted in filenames but retained in metadata.
- eCC versioning operates alongside CCP document versions; CCP changes may not always force eCC version changes unless XML content or IDs are affected.


## 19 SDC Security Considerations

While the TRG doesn’t prescribe a full security framework, it highlights key considerations for protecting SDC artifacts and data flows:

- **Package integrity:** sign or hash SDCPackages so implementers can verify that templates, maps, and demographic forms have not been tampered with.
- **Transport security:** use secure channels (TLS) when retrieving empty forms or submitting filled instances via REST/RFD to protect patient data in transit.
- **Access control:** Form Managers and Receivers should authenticate/authorize clients before distributing forms or accepting submissions; instance metadata often includes identifiers that should be treated as sensitive.
- **Data-at-rest protections:** when storing FDF-R documents or intermediate saves, ensure encryption/permissions align with clinical privacy policies (HIPAA, GDPR, etc.).
- **Auditability:** log package retrievals, submissions, and validation outcomes so organizations can trace who accessed which template and when data was transmitted.

These measures complement the structural guarantees of the SDC schema, helping organizations maintain confidentiality, integrity, and availability for forms and captured responses.


## 22 eCC Implementation Using Composite Keys (@IDs)

CAP eCC templates rely on composite identifiers to keep Question/ListItem semantics stable across releases. This chapter covers how @IDs tie together templates, terminologies, storage models, and transmissions.


### 22.1 Introduction

- Each eCC template item (Question, ListItem, etc.) carries an `@ID` that acts as a composite key when paired with the form’s `baseURI`. This allows implementers to unambiguously reference items across databases, mappings, and transmissions.


### 22.2 @IDs and Templates

- Template designers must maintain stable @IDs whenever semantics remain unchanged; when semantics change, deprecated items are replaced with new IDs (per Chapter 18’s versioning rules).
- These IDs are used in composite queries, SDC maps, and clinical registries to align responses with downstream analytics.


### 22.3 Data Storage, Terminologies, and Transmission

- **Terminology maps (22.3.1):** @IDs anchor mappings to ICD-O-3, SNOMED CT, and other code sets so updates can be managed centrally without losing alignment.
- **Data storage (22.3.2):** Databases should store both the captured values and their @IDs, enabling round-trips and cross-version comparisons.
- **Usage examples (22.3.3):** Composite keys support tasks like querying multiple template versions, merging eCC data with other systems, or generating comparative reports.
- **Other uses (22.3.4):** @IDs can drive business logic such as auto-population, UI customization, or linking to knowledge bases.
- **Transmission (22.3.5):** When sending data (e.g., via REST or other transports), include @IDs so receivers understand exactly which template items were answered—even if they don’t have the original FDF.


## 23 eCC Reference Implementation using HTML

The TRG includes an HTML-based reference implementation that demonstrates how to render SDC forms and auto-generate DEFs. This chapter captures lessons from that implementation for teams building their own UIs.


### 23.1 Automatic DEF Generation

- The reference implementation converts FormDesign XML into HTML controls automatically, relying on the structural conventions described throughout this notebook.
- It emphasizes consistent handling of Sections, Questions, and ListItems, including repeating blocks, activation logic, and reportText overrides.


### 23.1.1 Common Elements & Attributes

- The TRG provides a table of frequently used elements/attributes in the HTML implementation (e.g., how `title`, `reportText`, `selectionDisablesChildren`, `minCard`, `maxCard`, etc., map to specific UI behaviors).
- Builders can use this table as a checklist to ensure their rendering engine recognizes the core metadata needed for accurate behavior (activation, validation, reporting cues).
- Although the reference implementation targets HTML, the same mapping concepts apply to other UI frameworks—use the FormDesign metadata as the source of truth for layout and logic.
